In [ ]:
# The phoneme trainer: a base-sized ear on TIMIT, our recipe unchanged.
# Notebook settings that do not show in this file: internet ON, persistence
# "Files only" (it falls back to none on every Copy & Edit), pinned
# environment, T4 accelerator. Inputs: the code dataset and the corpus
# notebook, both named in the cell below.

# %% 1 -- what is actually mounted (the names are read, never guessed)
import os
for base, dirs, files in os.walk("/kaggle/input"):
    for name in files[:5]:
        print(base, name, os.path.getsize(os.path.join(base, name)))


In [ ]:
%%time
# %% 2 -- the code and the corpus
import shutil, subprocess
from pathlib import Path

CODE = Path("/kaggle/input/datasets/gilleslandrin/speakup-train")   # holds train/*.py
CORPUS = Path("/kaggle/input/notebooks/gilleslandrin/timit/timit.tar.gz")

WORK = Path("/kaggle/working")
source = CODE / "train" if (CODE / "train").is_dir() else CODE
(WORK / "train").mkdir(exist_ok=True)
copied = [shutil.copy(py, WORK / "train" / py.name) for py in source.glob("*.py")]
# A wrong mount path makes an empty glob: say it here, not three cells later.
assert copied, f"no .py under {source}"
print(copied)
(WORK / "tmp").mkdir(exist_ok=True)
# The archive stays an archive; extracting costs under a minute.
subprocess.run(["tar", "-xzf", str(CORPUS), "-C", str(WORK / "tmp")], check=True)


In [ ]:
%%time
# %% 3 -- the HF cache on /kaggle/temp (working's persistence loses the blobs).
# HF_HOME prefixes the subprocess and is never set in the notebook kernel.
# The vocabulary is the outgoing model's, which is what keeps the reading
# before and after a comparison at one alphabet.
import os, subprocess
ENV = {**os.environ, "HF_HOME": "/kaggle/temp/hf"}
subprocess.run(
    ["python", "-c",
     "from huggingface_hub import hf_hub_download; "
     "print(hf_hub_download('vitouphy/wav2vec2-xls-r-300m-timit-phoneme', 'vocab.json'))"],
    env=ENV, check=True)


In [ ]:
%%time
# %% 4 -- the decode proof, redone on the machine that will train
import os, subprocess
ENV = {**os.environ, "HF_HOME": "/kaggle/temp/hf"}
subprocess.run(["python", "train/manifest.py"],
               cwd="/kaggle/working", env=ENV, check=True)


In [ ]:
%%time
# %% 5 -- what a step costs on this machine, before committing to thirty epochs.
# The v3 notebook carried 1.00 s a step, 7.7 min an epoch and a 7943 MB peak on
# 15360; those are the 300 M network's numbers and say nothing about a third of
# it. This is the cell that produced them. It belongs in a draft session -- the
# long run belongs in Save & Run All, which survives a closed browser.
import os, shutil, subprocess, threading, time
from pathlib import Path

ENV = {**os.environ, "HF_HOME": "/kaggle/temp/hf"}
SMOKE = Path("/kaggle/working/tmp/train/smoke")
shutil.rmtree(SMOKE, ignore_errors=True)
STEPS = 20

peak, watching = 0, True

def watch():
    global peak
    while watching:
        seen = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True).stdout.split()
        peak = max([peak] + [int(value) for value in seen])
        time.sleep(1)

eye = threading.Thread(target=watch, daemon=True)
eye.start()
started = time.perf_counter()
subprocess.run(
    ["python", "-u", "train/train.py",
     "--encoder", "facebook/wav2vec2-base",
     "--unfreeze", "--checkpointing",
     "--lr", "1e-4", "--epochs", "30", "--batch", "8", "--accumulate", "2",
     "--warmup", "0.1", "--prior-weight", "0.0",
     "--steps", str(STEPS), "--out", str(SMOKE)],
    cwd="/kaggle/working", env=ENV, check=True)
spent = time.perf_counter() - started
watching = False

# 3696 utterances at batch 8 accumulated by 2: 231 optimiser steps an epoch.
per_step = spent / STEPS
print(f"\n{per_step:.2f} s a step, {231 * per_step / 60:.1f} min an epoch,"
      f" {30 * 231 * per_step / 3600:.1f} h for thirty")
print(f"{peak} MB peak (the session allows 12 h)")
shutil.rmtree(SMOKE, ignore_errors=True)


In [ ]:
# %% 6 -- unblock the output folder. train.py refuses to write into one that
# already holds checkpoints, which is what a run killed at epoch 12 leaves.
# This cell sits immediately before the one that trains: a sweep of
# /kaggle/working would carry off the code copied in cell 2.
import pathlib, shutil

out = pathlib.Path("/kaggle/working/tmp/train/runs-v4/v4-pw0.0")   # the run's --out
if out.exists():
    shutil.rmtree(out)
    print("removed:", out)


In [ ]:
%%time
# %% 7 -- the isolate: the v3 regime with the encoder alone changed.
#
# --encoder is the whole point. `facebook/wav2vec2-base` is 90 M trainable
# against the outgoing 315 M, self-supervised only, never fine-tuned to a task.
#
# --prior-weight 0.0 is deliberate and is NOT the script's default (0.3). The
# frequency penalty exists to move a sound's covered duration, and that extent
# has no consumer left (docs/analysis.md). It is now a lever that buys detection
# and sells the join, to be set when we know what we want to trade -- so the
# isolate holds it at zero, where `v3-pw0.0` sits and can be read against.
#
# Everything else is v3's, held identical on purpose: 30 epochs, batch 8,
# accumulate 2, lr 1e-4, warmup 0.1, --save-every 10 for epochs 009/019/029.
#
# The timings on file -- 1.00 s a step, 7.7 min an epoch, 7943 MB peak -- were
# measured on the 300 M network and do not transfer. A third of the parameters
# will be faster and lighter; nothing here has been timed, and the 12 h session
# cap was not close even then.
import os, subprocess
from pathlib import Path

ENV = {**os.environ, "HF_HOME": "/kaggle/temp/hf"}
OUT = Path("/kaggle/working/tmp/train/runs-v4/v4-pw0.0")
EPOCH = 462  # lots per epoch at batch 8: one loss line per epoch, not 462

run = subprocess.Popen(
    ["python", "-u", "train/train.py",
     "--encoder", "facebook/wav2vec2-base",
     "--unfreeze", "--checkpointing",
     "--lr", "1e-4", "--epochs", "30", "--batch", "8", "--accumulate", "2",
     "--warmup", "0.1", "--save-every", "10",
     "--prior-weight", "0.0",
     "--out", str(OUT)],
    cwd="/kaggle/working", env=ENV,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
seen = 0
for line in run.stdout:
    if " loss " in line:
        seen += 1
        if seen % EPOCH:
            continue
    print(line.rstrip(), flush=True)
if run.wait():
    raise SystemExit("the run failed")
print(sorted(e.name for e in OUT.glob("epoch-*")), flush=True)


In [ ]:
# %% 8 -- a version's output is all of /kaggle/working: keep only the runs
import shutil
shutil.rmtree("/kaggle/working/tmp/TIMIT", ignore_errors=True)
